In [ ]:
import os
import ast
from PIL import Image
import google.generativeai as genai  
from config import *
from plant_health_detection import *

### Agents Orcestration:

In [ ]:
model_name = 'gemini-3-flash-preview'
model = genai.GenerativeModel(model_name)


def planner_agent(model, plan_prompet, topic: str) -> list[str]:
    print("Planner Agent: Creating a research plan...")

    prompt = plan_prompet

    try:
        response = model.generate_content(prompt)
        plan = ast.literal_eval(response.text.strip())

        print("Plan created:")
        for i, q in enumerate(plan, 1):
            print(f"   {i}. {q}")
        return plan

    except Exception as e:
        print(f"Error in Planner Agent: {e}")
        return []



def search_agent(model, image_path: str, questions: str) -> str:
    # print(f"Search Agent: Researching question: '{questions}' for image '{image_path}'...")

    try:
        # Open the image
        query_image = Image.open(image_path).convert("RGB")

        # Compose a simple prompt asking Gemini to analyze image + question
        prompt = f"""
        You are an expert wheat plant pathologist in these diseases {list(diseases_and_description.keys())} and search agent. Examine the image and answer the following question:
        {questions}, Answer with clear and detailed observations based on visual inspection of the image and link with the known diseases and their descriptions.
        """

        # Generate content directly with the prompt and image
        response = model.generate_content([prompt, query_image])

        # print("   - Information found.")
        return response.text

    except Exception as e:
        print(f"Error in Search Agent: {e}")
        return ""



def synthesizer_agent(model, research_results: list) -> str:
    # print("You are an expert wheat plant pathologist and synthesizer Agent: Writing the final report...")

    research_notes = ""
    for question, data in research_results:
        research_notes += f"### Question: {question}\n### Research Data:\n{data}\n\n---\n\n"

    prompt = f"""
    You are an expert VINEYARD plant pathologist in these diseases {list(diseases_and_description.keys())} and synthesizer agent. Your task is to synthesize the provided research notes
    into the following structure ONLY:

    ["Disease or Pest Name or Healthy Plant (without any details, just the name)",
    "Severity Level (Low, Medium, High) (without any details, just the name)",
    "Probability of LLM prediction (0-100) (without any details, just the number)",
    "Recommended Actions"]

    ## Research Notes ##
    {research_notes}
    """

    try:
        response = model.generate_content(prompt)
        return response.text
    except Exception as e:
        print(f"Error in Synthesizer Agent: {e}")
        return "Error: Could not generate the final report."

genai.GenerativeModel(
    model_name='models/gemini-3-flash-preview',
    generation_config={},
    safety_settings={},
    tools=None,
    system_instruction=None,
    cached_content=None
)

### APHID:

In [6]:
simple_response_aphid = ask_gemini_about_image(
    "diseases\QA_aphid.jpg",
    SIMPLE_PROMPT,
    fewshot=False,
    train_dir=None,
    max_images_per_class=2,
    model_name="gemini-3-flash-preview"
)

detailed_response_aphid = ask_gemini_about_image(
    "diseases\QA_aphid.jpg",
    DETAILED_PROMPT,
    fewshot=False,
    train_dir=None,
    max_images_per_class=2,
    model_name="gemini-3-flash-preview"
)

quality_prompet_response_aphid = ask_gemini_about_image(
    "diseases\QA_aphid.jpg",
    QUALITY_DISEASE_CHECK_PROMPT,
    fewshot=False,
    train_dir=None,
    max_images_per_class=2,
    model_name="gemini-3-flash-preview"
)

simple_response_aphid, detailed_response_aphid, quality_prompet_response_aphid

('response:\nGenerateContentResponse(\n    done=True,\n    iterator=None,\n    result=protos.GenerateContentResponse({\n      "candidates": [\n        {\n          "content": {\n            "parts": [\n              {\n                "text": "Based on the image provided, here is the assessment of the plant\'s health:\\n\\n1.  **Plant type:** Grapevine (*Vitis* sp.).\\n2.  **Visible diseases or symptoms:** Heavy infestation of small black insects (aphids) on the undersides of the leaves, presence of ants tending to the insects, and leaf chlorosis (yellowing of the foliage).\\n3.  **Health status:** Not healthy; Aphid infestation."\n              }\n            ],\n            "role": "model"\n          },\n          "finish_reason": "STOP",\n          "index": 0\n        }\n      ],\n      "usage_metadata": {\n        "prompt_token_count": 1147,\n        "candidates_token_count": 95,\n        "total_token_count": 1655\n      },\n      "model_version": "gemini-3-flash-preview"\n    }),\

In [14]:
# Based on the image provided, here is the assessment of the plant's health:
# Plant type: Grapevine (Vitis sp.).
# Visible diseases or symptoms: Heavy infestation of small black insects (aphids) on the undersides of the leaves, presence of ants tending to the insects, and leaf chlorosis (yellowing of the foliage).
# Health status: Not healthy; Aphid infestation.
# -------------------------------------------


# "image_valid": true,
# "diseases": [
# "name": "aphid",
# "probability": 0.95
# "number_of_diseases": 1
# -------------------------------------------


# "quality_assessment": {
# "DEW_EFFECT": 0.0,
# "DISTANCE_EFFECT": 0.1,
# "LIGHT_EFFECT": 0.1,
# "GOOD_QUALITY": 0.8
# "image_valid": true,
# "name": "aphid",
# "probability": 0.95
# "number_of_diseases": 1

### Downey Mildew:

In [8]:
simple_response_downey_mildew = ask_gemini_about_image(
    "diseases\QA_grape_downey_mildew_2.jpg",
    SIMPLE_PROMPT,
    fewshot=False,
    train_dir=None,
    max_images_per_class=2,
    model_name="gemini-3-flash-preview"
)

detailed_response_downey_mildew = ask_gemini_about_image(
    "diseases\QA_grape_downey_mildew_2.jpg",
    DETAILED_PROMPT,
    fewshot=False,
    train_dir=None,
    max_images_per_class=2,
    model_name="gemini-3-flash-preview"
)

quality_prompet_response_downey_mildew = ask_gemini_about_image(
    "diseases\QA_grape_downey_mildew_2.jpg",
    QUALITY_DISEASE_CHECK_PROMPT,
    fewshot=False,
    train_dir=None,
    max_images_per_class=2,
    model_name="gemini-3-flash-preview"
)


simple_response_downey_mildew, detailed_response_downey_mildew, quality_prompet_response_downey_mildew

('response:\nGenerateContentResponse(\n    done=True,\n    iterator=None,\n    result=protos.GenerateContentResponse({\n      "candidates": [\n        {\n          "content": {\n            "parts": [\n              {\n                "text": "Based on the image provided:\\n\\n1. **Plant type:** Grape (Grapevine).\\n2. **Visible diseases or symptoms:** A white, powdery fungal growth is covering the grape berries and parts of the leaves.\\n3. **Health status:** Not healthy; the plant is infected with **Powdery Mildew**."\n              }\n            ],\n            "role": "model"\n          },\n          "finish_reason": "STOP",\n          "index": 0\n        }\n      ],\n      "usage_metadata": {\n        "prompt_token_count": 1143,\n        "candidates_token_count": 66,\n        "total_token_count": 1561\n      },\n      "model_version": "gemini-3-flash-preview"\n    }),\n)',
 'response:\nGenerateContentResponse(\n    done=True,\n    iterator=None,\n    result=protos.GenerateContent

In [13]:
# Based on the image provided:
# Plant type: Grape (Grapevine).
# Visible diseases or symptoms: A white, powdery fungal growth is covering the grape berries and parts of the leaves.
# Health status: Not healthy; the plant is infected with Powdery Mildew.
# -------------------------------------------


# "image_valid": true,
# "diseases": [
# "name": "downy_mildew",
# "probability": 0.95
# "number_of_diseases": 1
# -------------------------------------------


# "quality_assessment": {
# "DEW_EFFECT": 0.0,
# "DISTANCE_EFFECT": 0.05,
# "LIGHT_EFFECT": 0.05,
# "GOOD_QUALITY": 0.9
# "image_valid": true,
# "name": "downy_mildew",
# "probability": 0.98
# "number_of_diseases": 1

### Light Effects:

In [9]:
simple_response_light_effect = ask_gemini_about_image(
    "diseases\QA_grape_light_effect.jpg",
    SIMPLE_PROMPT,
    fewshot=False,
    train_dir=None,
    max_images_per_class=2,
    model_name="gemini-3-flash-preview"
)

detailed_response_light_effect = ask_gemini_about_image(
    "diseases\QA_grape_light_effect.jpg",
    DETAILED_PROMPT,
    fewshot=False,
    train_dir=None,
    max_images_per_class=2,
    model_name="gemini-3-flash-preview"
)

quality_prompet_response_light_effect = ask_gemini_about_image(
    "diseases\QA_grape_light_effect.jpg",
    QUALITY_DISEASE_CHECK_PROMPT,
    fewshot=False,
    train_dir=None,
    max_images_per_class=2,
    model_name="gemini-3-flash-preview"
)

simple_response_light_effect, detailed_response_light_effect, quality_prompet_response_light_effect

('response:\nGenerateContentResponse(\n    done=True,\n    iterator=None,\n    result=protos.GenerateContentResponse({\n      "candidates": [\n        {\n          "content": {\n            "parts": [\n              {\n                "text": "1. **Plant type:** Grape (Grapevine)\\n2. **Visible diseases or symptoms:** Small, dark, circular necrotic spots on several of the green berries.\\n3. **Health status:** Not healthy; Black Rot."\n              }\n            ],\n            "role": "model"\n          },\n          "finish_reason": "STOP",\n          "index": 0\n        }\n      ],\n      "usage_metadata": {\n        "prompt_token_count": 1135,\n        "candidates_token_count": 48,\n        "total_token_count": 1595\n      },\n      "model_version": "gemini-3-flash-preview"\n    }),\n)',
 'response:\nGenerateContentResponse(\n    done=True,\n    iterator=None,\n    result=protos.GenerateContentResponse({\n      "candidates": [\n        {\n          "content": {\n            "part

In [12]:
# 1. Plant type: Grape (Grapevine)
# 2. Visible diseases or symptoms: Small, dark, circular necrotic spots on several of the green berries.
# 3. Health status: Not healthy; Black Rot.
# -------------------------------------------


# "image_valid": true,
# "diseases": [
# "name": "Healthy",
# "probability": 0.98
# "number_of_diseases": 0
# -------------------------------------------


# "quality_assessment": {
# "DEW_EFFECT": 0.0,
# "DISTANCE_EFFECT": 0.05,
# "LIGHT_EFFECT": 0.1,
# "GOOD_QUALITY": 0.85
# "image_valid": true,
# "name": "Healthy",
# "probability": 1.0
# "number_of_diseases": 0

### Healthy:

In [11]:
simple_response_healthy = ask_gemini_about_image(
    "diseases\QA_healthy.jpg",
    SIMPLE_PROMPT,
    fewshot=False,
    train_dir=None,
    max_images_per_class=2,
    model_name="gemini-3-flash-preview"
)

detailed_response_healthy = ask_gemini_about_image(
    "diseases\QA_healthy.jpg",
    DETAILED_PROMPT,
    fewshot=False,
    train_dir=None,
    max_images_per_class=2,
    model_name="gemini-3-flash-preview"
)

quality_prompet_response_healthy = ask_gemini_about_image(
    "diseases\QA_healthy.jpg",
    QUALITY_DISEASE_CHECK_PROMPT,
    fewshot=False,
    train_dir=None,
    max_images_per_class=2,
    model_name="gemini-3-flash-preview"
)

simple_response_healthy, detailed_response_healthy, quality_prompet_response_healthy

('response:\nGenerateContentResponse(\n    done=True,\n    iterator=None,\n    result=protos.GenerateContentResponse({\n      "candidates": [\n        {\n          "content": {\n            "parts": [\n              {\n                "text": "Based on the image provided:\\n\\n1. **Plant type:** Mustard (Brassica species) used as a cover crop, and Grapevines (Vitis vinifera).\\n2. **Visible diseases or symptoms:** No visible symptoms of disease or pests. The mustard plants show vibrant green foliage and normal flowering, while the grapevines are in a natural state of dormancy.\\n3. **Health status:** Healthy."\n              }\n            ],\n            "role": "model"\n          },\n          "finish_reason": "STOP",\n          "index": 0\n        }\n      ],\n      "usage_metadata": {\n        "prompt_token_count": 1151,\n        "candidates_token_count": 85,\n        "total_token_count": 1639\n      },\n      "model_version": "gemini-3-flash-preview"\n    }),\n)',
 'response:\nGen

In [ ]:
# Based on the image provided:
# Plant type: Mustard (Brassica species) used as a cover crop, and Grapevines (Vitis vinifera).
# Visible diseases or symptoms: No visible symptoms of disease or pests. The mustard plants show vibrant green foliage and normal flowering, while the grapevines are in a natural state of dormancy.
# Health status: Healthy.
# -------------------------------------------


# "image_valid": false,
# "invalid_reasons": ["plant_too_far", "details_not_clear"],
# "diseases": [],
# "number_of_diseases": 0
# -------------------------------------------


# "quality_assessment": {
# "DEW_EFFECT": 0.0,
# "DISTANCE_EFFECT": 0.9,
# "LIGHT_EFFECT": 0.05,
# "GOOD_QUALITY": 0.05
# "image_valid": false,
# "number_of_diseases": 0

### Black Rot:

In [15]:
simple_response_black_rot = ask_gemini_about_image(
    "diseases\QA_grape_black_rot.jpeg",
    SIMPLE_PROMPT,
    fewshot=False,
    train_dir=None,
    max_images_per_class=2,
    model_name="gemini-3-flash-preview"
)

detailed_response_black_rot = ask_gemini_about_image(
    "diseases\QA_grape_black_rot.jpeg",
    DETAILED_PROMPT,
    fewshot=False,
    train_dir=None,
    max_images_per_class=2,
    model_name="gemini-3-flash-preview"
)

quality_prompet_response_black_rot = ask_gemini_about_image(
    "diseases\QA_grape_black_rot.jpeg",
    QUALITY_DISEASE_CHECK_PROMPT,
    fewshot=False,
    train_dir=None,
    max_images_per_class=2,
    model_name="gemini-3-flash-preview"
)

simple_response_black_rot, detailed_response_black_rot, quality_prompet_response_black_rot

('response:\nGenerateContentResponse(\n    done=True,\n    iterator=None,\n    result=protos.GenerateContentResponse({\n      "candidates": [\n        {\n          "content": {\n            "parts": [\n              {\n                "text": "Based on the image provided, here is the assessment of the plant\'s health:\\n\\n1. **Plant type:** Grapevine.\\n2. **Visible diseases or symptoms:** The grape berries exhibit sunken, circular, tan-to-gray spots with dark reddish-brown borders, characteristic of \\"bird\'s-eye\\" lesions. Some berries are shriveling and turning dark. There are also necrotic (dead) spots and yellowing on the leaves.\\n3. **Health status and disease name:** Not healthy. The plant is infected with **Grape Anthracnose** (also known as bird\'s-eye rot)."\n              }\n            ],\n            "role": "model"\n          },\n          "finish_reason": "STOP",\n          "index": 0\n        }\n      ],\n      "usage_metadata": {\n        "prompt_token_count": 1150

In [ ]:
# Based on the image provided, here is the assessment of the plant's health:
# Plant type: Grapevine.
# Visible diseases or symptoms: The grape berries exhibit sunken, circular, tan-to-gray spots with dark reddish-brown borders, characteristic of "bird's-eye" lesions. Some berries are shriveling and turning dark. There are also necrotic (dead) spots and yellowing on the leaves.
# Health status and disease name: Not healthy. The plant is infected with Grape Anthracnose (also known as bird's-eye rot).
# -------------------------------------------


# "image_valid": true,
# "diseases": [
# "name": "black_rot",
# "probability": 0.95
# "number_of_diseases": 1
# -------------------------------------------


# "quality_assessment": {
# "DEW_EFFECT": 0.0,
# "DISTANCE_EFFECT": 0.05,
# "LIGHT_EFFECT": 0.05,
# "GOOD_QUALITY": 0.9
# "image_valid": true,
# "name": "black_rot",
# "probability": 0.98
# "number_of_diseases": 1

---

In [11]:
questions = planner_agent(model, planner_prompet, topic)
image_path = "diseases/grape_downey_mildew_1.jpg"

answers_1 = []
for q in questions:
    answers_1.append(search_agent(model, image_path, q))
    
answers_2 = search_agent(model, image_path, questions)
final_answer_1_downey_mildew = synthesizer_agent(model, list(zip(questions, answers_1)))
final_answer_2_downey_mildew = synthesizer_agent(model, list(zip(questions, answers_2)))

final_answer_1_downey_mildew, final_answer_2_downey_mildew

Planner Agent: Creating a research plan...
Plan created:
   1. What are the primary visual symptoms visible on the grape leaves, stems, or berries in this image?
   2. Are there small, circular brown lesions with dark borders on the leaves or shrivelled, black mummified berries suggestive of Black Rot (Vineyard)?
   3. Do the leaves exhibit yellowish, translucent 'oil spots' on the upper surface or white, downy growth on the underside characteristic of Downy Mildew (Vineyard)?
   4. Are there visible clusters of small insects, sticky honeydew, or distorted/curled new growth indicating an infestation of Aphids (Vineyard)?
   5. Is the spatial distribution of the lesions or discoloration consistent with a spreading pathogen or localized pest damage?
   6. Does the overall plant vigor and foliage color suggest a healthy vineyard or one affected by the specific patterns of the target diseases?


('["Downy Mildew (Vineyard)",\n"High",\n"98",\n"Immediately apply targeted fungicides effective against oomycetes, such as copper-based products, phosphonates, or systemic fungicides like metalaxyl, ensuring thorough coverage of the clusters and foliage. Improve canopy management by leaf pulling and shoot thinning to increase airflow and reduce the humidity that facilitates pathogen sporulation. Remove and destroy heavily infected clusters and debris from the vineyard floor to reduce the primary inoculum. Monitor weather conditions closely, as warm and wet periods necessitate protective treatments to prevent further spread."]',
 '[\n    "Black Rot (Vineyard)",\n    "Medium",\n    "90",\n    "Prune and remove all mummified berries and infected canes to eliminate overwintering inoculum; apply targeted fungicides such as myclobutanil or mancozeb during the growing season; improve vineyard air circulation and sunlight penetration through proper canopy management; monitor for the appearance

In [12]:
questions = planner_agent(model, planner_prompet, topic)
image_path = "diseases/grape_black_rot.jpeg"

answers_1 = []
for q in questions:
    answers_1.append(search_agent(model, image_path, q))
    
answers_2 = search_agent(model, image_path, questions)
final_answer_1_black_rot = synthesizer_agent(model, list(zip(questions, answers_1)))
final_answer_2_black_rot = synthesizer_agent(model, list(zip(questions, answers_2)))

final_answer_1_black_rot, final_answer_2_black_rot

Planner Agent: Creating a research plan...
Plan created:
   1. What are the primary visual anomalies or symptoms visible on the leaf, stem, or fruit surfaces in this vineyard image?
   2. Are there clusters of small, soft-bodied insects on the shoots or leaf undersides, potentially accompanied by leaf curling or sticky residue characteristic of Aphids (Vineyard)?
   3. Are there distinct circular, reddish-brown lesions with tiny black fruiting bodies (pycnidia) on the leaves, or are grapes shriveled into hard black mummies consistent with Black Rot (Vineyard)?
   4. Do the leaves show translucent, yellowish 'oil spots' on the upper surface or white, cottony fungal growth on the underside indicative of Downy Mildew (Vineyard)?
   5. Is the spatial distribution and morphology of the symptoms consistent with the specific patterns of dict_keys(['Aphids (Vineyard)', 'Black Rot (Vineyard)', 'Downy Mildew (Vineyard)']) rather than general nutrient deficiencies or mechanical damage?
   6. Base

('["Black Rot (Vineyard)",\n"High",\n"95",\n"Remove and destroy all infected fruit clusters and mummified berries to reduce overwintering inoculum; apply targeted fungicides such as DMIs, strobilurins, or copper-based sprays during the critical period from bloom to veraison; improve vineyard sanitation by pruning out infected canes; and enhance canopy airflow through strategic leaf pulling to reduce the moisture that favors fungal germination."]',
 '["Healthy Plant",\n"Low",\n"100",\n"Conduct regular vineyard scouting and monitoring every 7–10 days to identify early signs of pest or disease pressure. Maintain proper canopy management through pruning and leaf thinning to enhance airflow and reduce microclimate humidity. Ensure the vineyard remains free of plant debris and maintain a consistent preventative maintenance schedule."]')

In [13]:
questions = planner_agent(model, planner_prompet, topic)
image_path = "diseases/grape_light_effect.jpg"

answers_1 = []
for q in questions:
    answers_1.append(search_agent(model, image_path, q))
    
answers_2 = search_agent(model, image_path, questions)
final_answer_1_black_rot = synthesizer_agent(model, list(zip(questions, answers_1)))
final_answer_2_black_rot = synthesizer_agent(model, list(zip(questions, answers_2)))

final_answer_1_black_rot, final_answer_2_black_rot

Planner Agent: Creating a research plan...
Plan created:
   1. - Are there visible discolorations, lesions, or textural abnormalities on the leaves, stems, or fruit in the image?
   2. - Do the leaf symptoms present as circular brown spots with dark margins and tiny black dots, which are indicative of Black Rot (Vineyard)?
   3. - Are there yellowish-green 'oil spots' on the upper leaf surface or white, downy fungal growth on the leaf underside consistent with Downy Mildew (Vineyard)?
   4. - Are there clusters of small, soft-bodied insects, leaf curling, or sticky honeydew residue characteristic of Aphids (Vineyard)?
   5. - Is the spatial distribution of the symptoms concentrated on new growth, scattered across mature leaves, or localized to the fruit clusters?
   6. - Do the observed visual patterns align with the specific diagnostic features of dict_keys(['Aphids (Vineyard)', 'Black Rot (Vineyard)', 'Downy Mildew (Vineyard)']) or does the plant appear healthy and asymptomatic?


('["Healthy Plant",\n"Low",\n"90",\n"Conduct regular visual inspections of leaf surfaces and fruit clusters for early signs of necrotic spots, \'oil spots,\' or insect activity. Maintain proper canopy management to ensure adequate airflow and sunlight penetration, which reduces moisture and discourages fungal pathogens like Black Rot and Downy Mildew. Implement a preventative integrated pest management (IPM) program, including the application of labeled fungicides during high-risk environmental periods."]',
 '["Black Rot (Vineyard)", "Medium", "85", "Remove and destroy all mummified berries and infected canes during the dormant season to eliminate overwintering inoculum; apply preventative fungicides such as myclobutanil or mancozeb starting at early bloom and continuing until berries reach 10-15% sugar content; implement canopy management practices like shoot thinning and leaf pulling to increase air circulation and reduce leaf wetness duration."]')